# LLM as a Judge to Detect Wrong Answers with kluster.ai

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluster-ai/klusterai-cookbook/blob/main/examples/llm-as-a-judge.ipynb)

Welcome to the LLM as a Judge Notebook!

Large language models (LLMs) are powerful tools for generating predictions, answering questions, and solving a variety of tasks. However, assessing the quality of their outputs often requires human judgment. When using a batch API to generate large volumes of predictions, manually evaluating each response can be challenging, if not impractical. This notebook introduces the concept of using LLMs themselves as judges to evaluate the outputs of other LLMs using <a href="https://kluster.ai/" target="_blank">kluster.ai</a> Batch API. By automating the evaluation process, we can gain insights into both the accuracy and quality of model-generated responses.

This notebook guides you through evaluating LLM-generated answers using another LLM as a judge. 
1. **Answer Initial Question:** An LLM creates four distinct answers to a single question—one correct response and three artificially altered versions, each containing different types of errors.
2. **Evaluate Responses:** Submit these predictions to another LLM, which will act as the judge, evaluating the quality and providing feedback on each answer.


## Config

Enter your personal kluster.ai API key (make sure it has no blank spaces). Remember to <a href="https://platform.kluster.ai/signup" target="_blank">sign up</a> if you don't have one yet.

In [52]:
from getpass import getpass
# Enter you personal kluster.ai API key (make sure in advance it has no blank spaces)
api_key = getpass("Enter your kluster.ai API key: ")

Enter your kluster.ai API key:  ········


## Setup

In [53]:
%pip install OpenAI

Note: you may need to restart the kernel to use updated packages.


In [54]:
import os
import urllib.request
import pandas as pd
import numpy as np
import random
import requests
from openai import OpenAI
import time
import json
from IPython.display import clear_output, display

pd.set_option('display.max_columns', 1000, 'display.width', 1000, 'display.max_rows',1000, 'display.max_colwidth', 500)

In [55]:
# Set up the client
client = OpenAI(
    base_url="https://api.kluster.ai/v1",
    api_key=api_key,
)

## 1. Answer a Simple Question

In this first part of the process, we will use an LLM to generate answers to a simple question. For this question, the model will be asked to produce three variations of its response.

This setup allows us to create a diverse range of answers for each question, intentionally varying their quality. These variations will serve as input for the next stage, where another model, acting as the judge, will evaluate the responses to determine which are good, which are flawed, and why.

In [35]:
df = pd.DataFrame({
    "question": [
        "What is 7777 + 3333?",
        "What is the smallest real number x in the domain of the function g(x) = sqrt((x − 3)^2 − (x − 8)^2) ?",
        "Which weighs more, a pound of water, two pounds of bricks, a pound of feathers, or three pounds of air?",
        "Sally (a girl) has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?"
    ]})

Now, using <a href="https://kluster.ai/" target="_blank">kluster.ai</a>'s Batch API we’ll create the batch tasks, generate the batch file, and upload it to the kluster.ai. Once uploaded, we’ll wait for the job to complete and retrieve the results. This process handles the generation of answers from the first model and prepares us for the evaluation step.

#### Create the Batch File

In [38]:
def create_tasks(df, task_type, system_prompt, model):
    tasks = []
    for index, row in df.iterrows():
        if task_type == 'assistant':
            content = row['question']
        elif task_type == 'judge':
            content = f'''
            User question: {row['question']}. \n\nLLM answer: {row['answer']}
            '''

        task = {
            "custom_id": f"{task_type}-{index}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "temperature": 0.5,
                "response_format": {"type": "json_object"},
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": content},
                ],
            }
        }
        tasks.append(task)
    return tasks

def save_tasks(tasks, task_type):
    filename = f"batch_tasks_{task_type}.jsonl"
    with open(filename, 'w') as file:
        for task in tasks:
            file.write(json.dumps(task) + '\n')
    return filename

In [39]:
ASSISTANT_PROMPT = '''
    Provide an answer to the question asked.
    '''

task_list = create_tasks(df, task_type='assistant', system_prompt=ASSISTANT_PROMPT, model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")
filename = save_tasks(task_list, task_type='assistant')

#### Upload Batch File to kluster.ai

In [40]:
def create_batch_job(file_name):
    print(f"Creating batch job for {file_name}")
    batch_file = client.files.create(
        file=open(file_name, "rb"),
        purpose="batch"
    )

    batch_job = client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    return batch_job

job = create_batch_job(filename)

Creating batch job for batch_tasks_assistant.jsonl


#### Check Job progress

In [41]:
def parse_json_objects(data_string):
    if isinstance(data_string, bytes):
        data_string = data_string.decode('utf-8')

    json_strings = data_string.strip().split('\n')
    json_objects = []

    for json_str in json_strings:
        try:
            json_obj = json.loads(json_str)
            json_objects.append(json_obj)
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON: {e}")

    return json_objects

def monitor_job_status(client, job_id, task_type):
    all_completed = False

    while not all_completed:
        all_completed = True
        output_lines = []

        updated_job = client.batches.retrieve(job_id)

        if updated_job.status.lower() != "completed":
            all_completed = False
            completed = updated_job.request_counts.completed
            total = updated_job.request_counts.total
            output_lines.append(f"{task_type.capitalize()} job status: {updated_job.status} - Progress: {completed}/{total}")
        else:
            output_lines.append(f"{task_type.capitalize()} job completed!")

        # Clear the output and display updated status
        clear_output(wait=True)
        for line in output_lines:
            display(line)

        if not all_completed:
            time.sleep(10)

monitor_job_status(client=client, job_id=job.id, task_type='assistant')

'Assistant job completed!'

#### Get the results

In [42]:
batch_job = client.batches.retrieve(job.id)
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content
results = parse_json_objects(result)
answers = []

for res in results:
    task_id = res['custom_id']
    result = res['response']['body']['choices'][0]['message']['content']
    answers.append(result) 

df['answer'] = answers

In [44]:
df.head()

,question,answer
0,What is 7777 + 3333?,"To find the sum, I will add 7777 and 3333.\n\n7777 + 3333 = 11110."
1,What is the smallest real number x in the domain of the function g(x) = sqrt((x − 3)^2 − (x − 8)^2) ?,"To find the smallest real number x in the domain of the function g(x), we need to find the values of x for which the expression inside the square root is non-negative.\n\nThe expression inside the square root is (x − 3)^2 − (x − 8)^2. Expanding this expression, we get:\n\n(x − 3)^2 − (x − 8)^2 = (x^2 - 6x + 9) - (x^2 - 16x + 64)\n= -6x + 9 + 16x - 64\n= 10x - 55\n\nNow, we need to find the values of x for which 10x - 55 ≥ 0.\n\n10x - 55 ≥ 0\n10x ≥ 55\nx ≥ 55/10\nx ≥ 5.5\n\nSo, the smallest r..."
2,"Which weighs more, a pound of water, two pounds of bricks, a pound of feathers, or three pounds of air?","To solve this, let's break it down:\n\n1. A pound of water weighs one pound.\n2. Two pounds of bricks weigh two pounds.\n3. A pound of feathers weighs one pound.\n4. Three pounds of air weigh three pounds.\n\nSince a pound of water, a pound of feathers, and three pounds of air all weigh the same amount (1 pound or 3 pounds), they have the same weight.\n\nHowever, two pounds of bricks weigh more than all the other options."
3,Sally (a girl) has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?,"Since Sally has 3 brothers, and each brother has 2 sisters, we might initially think that Sally has 6 sisters (3 brothers * 2 sisters). However, this would count Sally herself as one of her brothers' sisters.\n\nTo find the correct number of sisters Sally has, we need to consider the total number of sisters her brothers have, and then subtract Sally herself. \n\nSince each brother has 2 sisters, Sally's 3 brothers have a total of 6 sisters. However, Sally is one of those 6 sisters (she is on..."


## 2. Evaluate the answers

In this section, we will use our model in a new role—as a judge. The model will evaluate the response it previously generated, analyzing its quality and providing a score. This step helps us assess how well the model performed in generating the original response, offering valuable insights into its effectiveness.

In [46]:
JUDGE_PROMPT = '''
    You are an evaluator tasked with assessing the quality of a response provided by an expert to a specific user question. 
    Generate a score from 1 to 5 based on relevance, accuracy and clarity and provide a brief explanation for your score. 
    Output your evaluation in this format: “Score: X/5, Explanation: [Your explanation]“
    Use no more than 50 words in the explanation.
    '''

Now, we’ll erase the information about which answer was generated by adding some invented details

In [47]:
task_list = create_tasks(df, task_type='judge', system_prompt=JUDGE_PROMPT, model = "klusterai/Meta-Llama-3.1-405B-Instruct-Turbo")
filename = save_tasks(task_list, task_type='judge')
job = create_batch_job(filename)
monitor_job_status(client=client, job_id=job.id, task_type='judge')

'Judge job completed!'

In [51]:
batch_job = client.batches.retrieve(job.id)
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content
results = parse_json_objects(result)
evals = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    evals.append(result) 
    question = df.iloc[int(index)]['question']
    answer = df.iloc[int(index)]['answer']
    print(f'\n -------------------------- \n')
    print(f"Task ID: {task_id}. \n\nQUESTION: {question} \n\nLLM ANSWER: {answer} \n\nEVALUATION: {result}")

df['evaluation'] = evals


 -------------------------- 

Task ID: judge-0. 

QUESTION: What is 7777 + 3333? 

LLM ANSWER: To find the sum, I will add 7777 and 3333.

7777 + 3333 = 11110. 

EVALUATION: Score: 5/5, Explanation: The response is relevant, accurate, and clear. It directly addresses the user's question, performs the calculation correctly, and provides a straightforward answer, making it easy for the user to understand the solution.

 -------------------------- 

Task ID: judge-1. 

QUESTION: What is the smallest real number x in the domain of the function g(x) = sqrt((x − 3)^2 − (x − 8)^2) ? 

LLM ANSWER: To find the smallest real number x in the domain of the function g(x), we need to find the values of x for which the expression inside the square root is non-negative.

The expression inside the square root is (x − 3)^2 − (x − 8)^2. Expanding this expression, we get:

(x − 3)^2 − (x − 8)^2 = (x^2 - 6x + 9) - (x^2 - 16x + 64)
= -6x + 9 + 16x - 64
= 10x - 55

Now, we need to find the values of x for w